# Analysis Report

## 1 - Setup and Data Loading

In [ ]:
from __future__ import annotations

import warnings

try:
    import matplotlib_inline
    matplotlib_inline.backend_inline.set_matplotlib_formats("svg", "pdf")
except ImportError:
    pass

warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, HTML

In [ ]:
from matplotlib.figure import Figure

# ── Visual Configuration ──────────────────────────────────────────────

sns.set_theme(
    style="whitegrid",
    rc={
        "axes.facecolor": "#F8F9FA",
        "grid.color": "#E9ECEF",
        "font.size": 13,
        "axes.labelsize": 15,
        "axes.titlesize": 18,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
        "figure.dpi": 150,
        "savefig.dpi": 300,
    },
)

# Grammar color palette (primary comparison axis)
GRAMMAR_PALETTE: dict[str, str] = {
    "semgir":       "#4C72B0",
    "squall":       "#DD8452",
    "graphq_tree":  "#55A868",
}

# Model markers (secondary comparison axis)
MODEL_MARKERS: list[str] = ["o", "s", "D", "^", "v", "P", "X"]

# Output directory for saved figures
FIGURES_DIR = Path("outputs/analysis/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def get_grammar_color(grammar: str) -> str:
    """Return the color for a grammar, with a fallback for ablation variants."""
    key = grammar.lower()
    if key in GRAMMAR_PALETTE:
        return GRAMMAR_PALETTE[key]
    for base, color in GRAMMAR_PALETTE.items():
        if base in key:
            return color
    # Fallback: pick from seaborn's default
    return sns.color_palette("muted").as_hex()[len(GRAMMAR_PALETTE) % 10]

def save_fig(fig: Figure, name: str) -> None:
    """Save a figure as SVG, PDF, and PNG."""
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{name}.{ext}", bbox_inches="tight")

In [ ]:
# ── Data Loading ─────────────────────────────────────────────────────
# Point this to the metrics output directory for the run you want to analyze.

METRICS_DIR = Path("outputs/metrics")  # adjust if needed

# Auto-detect the latest run directory
run_dirs = sorted([d for d in METRICS_DIR.iterdir() if d.is_dir()])
if not run_dirs:
    raise FileNotFoundError(f"No run directories found in {METRICS_DIR}")
RUN_DIR = run_dirs[-1]
print(f"Analyzing run: {RUN_DIR.name}")

df_query   = pd.read_csv(RUN_DIR / "query_level_metrics.csv")
df_summary = pd.read_csv(RUN_DIR / "model_summary_metrics.csv")
df_errors  = pd.read_csv(RUN_DIR / "error_distribution.csv")
df_sig     = pd.read_csv(RUN_DIR / "significance_tests.csv")

# Derive ordered lists
grammars = sorted(df_query["Grammar"].unique())
models   = sorted(df_query["Model"].unique())

print(f"Grammars: {grammars}")
print(f"Models:   {models}")
print(f"Queries:  {df_query['QID'].nunique()}")

## 2 - Performance and Reliability Metrics

### 2.1 Summary Table

In [ ]:
# Pivot the model summary into a clean table for display
display_cols = [c for c in df_summary.columns if c not in ("Grammar", "Model")]
summary_pivot = df_summary.set_index(["Grammar", "Model"])[display_cols].round(4)
display(summary_pivot)

### 2.2 Success Rate & Hypothesis Coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, metric_col, title in zip(
    axes,
    ["Avg Success Rate", "Avg Hypothesis Coverage"],
    ["Success Rate", "Hypothesis Coverage"],
):
    x = np.arange(len(models))
    width = 0.8 / len(grammars)

    for i, grammar in enumerate(grammars):
        subset = df_summary[df_summary["Grammar"] == grammar].set_index("Model").reindex(models)
        vals = subset[metric_col].fillna(0).values
        ax.bar(
            x + i * width,
            vals,
            width,
            label=grammar,
            color=get_grammar_color(grammar),
            edgecolor="black",
            linewidth=0.5,
        )

    ax.set_ylabel(title, fontweight="bold")
    ax.set_title(title, fontweight="bold", pad=12)
    ax.set_xticks(x + width * (len(grammars) - 1) / 2)
    ax.set_xticklabels(models, rotation=15, ha="right", fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.legend(title="Grammar", frameon=True, fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax, left=True, bottom=False)

fig.tight_layout()
save_fig(fig, "success_rate_coverage")
plt.show()

### 2.3 Capability Curve (Pass@k)

In [ ]:
# Detect k values from columns
k_cols = sorted([c for c in df_summary.columns if c.startswith("Pass@")], key=lambda c: int(c.split("@")[1]))
k_values = [int(c.split("@")[1]) for c in k_cols]

n_models = len(models)
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 5), sharey=True, squeeze=False)
axes = axes[0]

for ax, model in zip(axes, models):
    for grammar in grammars:
        row = df_summary[(df_summary["Model"] == model) & (df_summary["Grammar"] == grammar)]
        if row.empty:
            continue
        vals = [row[col].values[0] for col in k_cols]
        ax.plot(
            k_values, vals,
            marker="o", linewidth=2, markersize=6,
            label=grammar, color=get_grammar_color(grammar),
        )

    ax.set_title(model, fontweight="bold", pad=10)
    ax.set_xlabel("k", fontweight="bold")
    ax.set_xticks(k_values)
    ax.set_xticklabels(k_values)
    ax.set_ylim(-0.02, 1.05)
    ax.legend(title="Grammar", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    sns.despine(ax=ax, left=True, bottom=False)

axes[0].set_ylabel("Pass@k", fontweight="bold")
fig.suptitle("Capability Curve — Pass@k", fontweight="bold", fontsize=18, y=1.02)
fig.tight_layout()
save_fig(fig, "pass_at_k")
plt.show()

### 2.4 Consistency Curve (Pass^k)

In [ ]:
pk_cols = sorted([c for c in df_summary.columns if c.startswith("Pass^")], key=lambda c: int(c.split("^")[1]))
pk_values = [int(c.split("^")[1]) for c in pk_cols]

fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 5), sharey=True, squeeze=False)
axes = axes[0]

for ax, model in zip(axes, models):
    for grammar in grammars:
        row = df_summary[(df_summary["Model"] == model) & (df_summary["Grammar"] == grammar)]
        if row.empty:
            continue
        vals = [row[col].values[0] for col in pk_cols]
        ax.plot(
            pk_values, vals,
            marker="s", linewidth=2, markersize=6, linestyle="--",
            label=grammar, color=get_grammar_color(grammar),
        )

    ax.set_title(model, fontweight="bold", pad=10)
    ax.set_xlabel("k", fontweight="bold")
    ax.set_xticks(pk_values)
    ax.set_xticklabels(pk_values)
    ax.set_ylim(-0.02, 1.05)
    ax.legend(title="Grammar", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    sns.despine(ax=ax, left=True, bottom=False)

axes[0].set_ylabel("Pass^k", fontweight="bold")
fig.suptitle("Consistency Curve — Pass^k", fontweight="bold", fontsize=18, y=1.02)
fig.tight_layout()
save_fig(fig, "pass_power_k")
plt.show()

### 2.5 Statistical Significance Table

In [ ]:
def highlight_sig(val):
    """Bold p-values below 0.05."""
    try:
        if float(val) < 0.05:
            return "font-weight: bold; color: #c0392b"
    except (ValueError, TypeError):
        pass
    return ""


styled_sig = (
    df_sig.style
    .map(highlight_sig, subset=["p-value"])
    .apply(
        lambda col: [
            "font-weight: bold" if v else "" for v in col
        ],
        subset=["Significant (p<0.05)"],
    )
    .format({"p-value": "{:.4f}"})
)
display(styled_sig)

## 3 - Quantitative Complexity Analysis

### 3.1 Macro Efficiency Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, metric_col, title, ylabel in zip(
    axes,
    ["Avg Completion Tokens", "Avg Inference Time (s)"],
    ["Avg Completion Tokens", "Avg Inference Time"],
    ["Tokens", "Seconds"],
):
    x = np.arange(len(models))
    width = 0.8 / len(grammars)

    for i, grammar in enumerate(grammars):
        subset = df_summary[df_summary["Grammar"] == grammar].set_index("Model").reindex(models)
        vals = subset[metric_col].fillna(0).values
        ax.bar(
            x + i * width,
            vals,
            width,
            label=grammar,
            color=get_grammar_color(grammar),
            edgecolor="black",
            linewidth=0.5,
        )

    ax.set_ylabel(ylabel, fontweight="bold")
    ax.set_title(title, fontweight="bold", pad=12)
    ax.set_xticks(x + width * (len(grammars) - 1) / 2)
    ax.set_xticklabels(models, rotation=15, ha="right", fontweight="bold")
    ax.legend(title="Grammar", frameon=True, fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax, left=True, bottom=False)

fig.tight_layout()
save_fig(fig, "efficiency_comparison")
plt.show()

### 3.2 Query-Level Variance

In [ ]:
for grammar in grammars:
    df_g = df_query[df_query["Grammar"] == grammar].copy()
    df_g = df_g.sort_values("QID")

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for ax, metric, ylabel in zip(
        axes,
        ["Inference Time (s)", "Completion Tokens"],
        ["Seconds", "Tokens"],
    ):
        for j, model in enumerate(models):
            df_m = df_g[df_g["Model"] == model]
            marker = MODEL_MARKERS[j % len(MODEL_MARKERS)]
            ax.plot(
                range(len(df_m)),
                df_m[metric].values,
                marker=marker,
                markersize=5,
                linewidth=1.2,
                label=model,
                alpha=0.85,
            )

        ax.set_xlabel("Query Index", fontweight="bold")
        ax.set_ylabel(ylabel, fontweight="bold")
        ax.set_title(f"{metric} per Query", fontweight="bold", pad=10)
        ax.legend(title="Model", fontsize=9, loc="best")
        ax.grid(axis="y", linestyle="--", alpha=0.7)
        sns.despine(ax=ax, left=True, bottom=False)

    fig.suptitle(
        f"Query-Level Variance — {grammar}",
        fontweight="bold", fontsize=18, y=1.02,
    )
    fig.tight_layout()
    save_fig(fig, f"query_variance_{grammar}")
    plt.show()

## 4 - Qualitative Error Analysis

### 4.1 Normalized Error Distribution

In [ ]:
if df_errors.empty:
    print("No error data available.")
else:
    # Create a combined label for x-axis
    df_err = df_errors.copy()
    df_err["Group"] = df_err["Grammar"] + " | " + df_err["Model"]

    # Pivot: rows = Group, columns = Error Category, values = Percentage
    pivot = df_err.pivot_table(
        index="Group",
        columns="Error Category",
        values="Percentage",
        aggfunc="sum",
        fill_value=0,
    )

    # Sort groups for consistent ordering
    pivot = pivot.reindex(sorted(pivot.index))

    # Color palette for error categories
    n_cats = len(pivot.columns)
    err_palette = sns.color_palette("Set2", n_cats)

    fig, ax = plt.subplots(figsize=(max(12, len(pivot) * 1.5), 7))

    pivot.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=err_palette,
        edgecolor="white",
        linewidth=0.5,
        width=0.7,
    )

    ax.set_ylabel("Proportion (%)", fontweight="bold")
    ax.set_xlabel("")
    ax.set_title(
        "Normalized Error Distribution (100%)",
        fontweight="bold", pad=12, fontsize=18,
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=11)
    ax.set_ylim(0, 105)
    ax.legend(
        title="Error Category",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=True,
        fontsize=10,
    )
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    ax.grid(axis="x", visible=False)
    sns.despine(ax=ax, left=True, bottom=False)

    fig.tight_layout()
    save_fig(fig, "error_distribution_normalized")
    plt.show()

### 4.2 Error Tables

In [ ]:
if df_errors.empty:
    print("No error data available.")
else:
    err_table = df_errors.pivot_table(
        index=["Grammar", "Model"],
        columns="Error Category",
        values=["Count", "Percentage"],
        aggfunc="sum",
        fill_value=0,
    )
    display(err_table)